# BLOCK-T70 - Dome Handover

This notebook creates a sequence of movements we want to execute to partially open and close the dome shutter during the handover procedure.

In [ ]:
import numpy as np
import os

from lsst.ts.observing import ObservingBlock, ObservingScript

In [ ]:
name = "BLOCK-T70"
program = "BLOCK-T70"
reason = "BLOCK-T70"
constraints = []
scripts = []

try:
    output_folder = (
        os.environ["TS_CONFIG_OCS_DIR"] + "/Scheduler/observing_blocks_maintel"
    )
except KeyError:
    warnings.warn(
        "The environment variable 'TS_CONFIG_OCS_DIR' is not set. Using default folder 'output_blocks'."
    )
output_folder = "output_blocks"


In [ ]:
def build_configuration_schema(block_number, properties):
    """
    Builds a configuration schema string for a given BLOCK and configurable
    properties.

    Parameters
    ----------
    block_number :
        The BLOCK number to include in the title and description.
    properties : dict
        A dictionary where each key is a property name, and each value is a
        dictionary with keys 'description', 'type', and optionally 'default'.

    Returns
    -------
        A formatted configuration schema string.
    """

    # Define the base schema with the BLOCK number
    configuration_schema = (
        "$schema: http://json-schema.org/draft-07/schema#\n"
        f"title: BLOCK-{block_number} configuration\n"
        f"description: Configuration for BLOCK-{block_number}.\n"
        "type: object\n"
        "properties:\n"
    )

    # Add each property to the schema
    for prop_name, prop_details in properties.items():
        configuration_schema += f"  {prop_name}:\n"
        configuration_schema += f'    description: {prop_details["description"]}\n'
        configuration_schema += f'    type: {prop_details["type"]}\n'
        if "default" in prop_details:
            # Add quotes around the default value if it's a string
            default_value = prop_details["default"]
            if prop_details["type"] == "string":
                default_value = f'"{default_value}"'
            configuration_schema += f"    default: {default_value}\n"

    return configuration_schema

Define the configurable properties that we will use in the configuration schema

In [ ]:
# Custom properties for the BLOCK
properties = {
    "sleep_for": {
        "description": "Duration of the sleep between opening and cllosing the shutter.",
        "type": "number",
        "default": "30",
    },
}

block_number = name.split("-")[-1]
configuration_schema = build_configuration_schema(block_number, properties)
print(configuration_schema)

In [ ]:
# Open shutter block
open_shutter = ObservingScript(
    name="run_command.py",
    standard=True,
    parameters=dict(
        component="MTDome",
        cmd="openShutter",
    ),
)

# Sleep for a specified duration
sleep_for = ObservingScript(
    name="sleep.py",
    standard=True,
    parameters=dict(
        sleep_for="$sleep_for",
    ),
)

# Stop the dome shutter
stop_dome_shutter = ObservingScript(
    name="run_command.py",
    standard=True,
    parameters=dict(
        component="MTDome",
        cmd="stop",
        parameters=dict(
            subSystemIds=0x4,
        ),
    ),
)

# Sleep for 1 s
sleep1s = ObservingScript(
    name="sleep.py",
    standard=True,
    parameters=dict(
        sleep_for=1,
    ),
)

# Close shutter block
close_shutter = ObservingScript(
    name="run_command.py",
    standard=True,
    parameters=dict(
        component="MTDome",
        cmd="closeShutter",
    ),
)

scripts.append(open_shutter)
scripts.append(sleep_for)
scripts.append(stop_dome_shutter)
scripts.append(sleep1s)
scripts.append(close_shutter)

In [ ]:
block = ObservingBlock(
    name=name,
    program=program,
    configuration_schema=configuration_schema,
    scripts=scripts,
)

In [ ]:
# Write the block to JSON file

os.makedirs(output_folder, exist_ok=True)
output_path = f"{output_folder}/{name}.json"

with open(output_path, "w") as file:
    file.write(block.model_dump_json(indent=4))